
# VARNN Pipeline cho Dự báo `cpi_mom_inflation`

Notebook này xây dựng một **pipeline hoàn chỉnh** cho bài toán dự báo chuỗi thời gian đa biến với **VARNN = VAR-NN (Vector Autoregressive Neural Network)**, tập trung vào mục tiêu dự báo `cpi_mom_inflation` 1 bước tới.

## Nội dung chính
- Tiền xử lý dữ liệu chuỗi thời gian theo **không rò rỉ dữ liệu (no leakage)**
- Xử lý riêng `ppi_qoq`: **đánh dấu là biến quý** và **tự động loại bỏ nếu tương quan thấp**
- Kiểm định tính dừng (ADF), biến đổi dữ liệu theo từng nhóm biến
- Chuẩn hóa bằng **StandardScaler**
- Chọn **lag** bằng **VAR + AIC/BIC**
- Tạo **supervised matrix**
- Huấn luyện **VARNN** bằng **TimeSeriesSplit cross-validation**
- Tối ưu siêu tham số
- Chẩn đoán **underfitting / overfitting** và **tự động khắc phục**
- **Chỉ in kết quả ra màn hình**, không xuất file/hình ảnh

> **Ghi chú:** Notebook mặc định dự báo **một biến mục tiêu** là `cpi_mom_inflation`, nhưng có thể mở rộng sang multi-output nếu cần.



## 1. Công thức và giả thuyết sử dụng

### 1.1 Chuẩn hóa StandardScaler

Với mỗi biến $x$, StandardScaler biến đổi về:

$$
z = \frac{x - \mu}{\sigma}
$$

trong đó:
- $\mu$: trung bình của biến trên **tập train**
- $\sigma$: độ lệch chuẩn của biến trên **tập train**

### 1.2 Kiểm định ADF (Augmented Dickey-Fuller)

Giả thuyết kiểm định:

$$
H_0: \text{Chuỗi có nghiệm đơn vị (unit root), không dừng}
$$

$$
H_1: \text{Chuỗi dừng}
$$

Quy tắc quyết định với mức ý nghĩa $\alpha$:

$$
\text{Nếu } p\text{-value} < \alpha \Rightarrow \text{bác bỏ } H_0
$$

### 1.3 Kiểm định tương quan cho `ppi_qoq`

Để đánh giá mối liên hệ tuyến tính với biến mục tiêu, dùng Pearson correlation:

$$
H_0: \rho = 0
$$

$$
H_1: \rho \neq 0
$$

Nếu $|r|$ nhỏ và $p$-value lớn, có thể cân nhắc loại biến `ppi_qoq`.

### 1.4 Ma trận giám sát (Supervised Matrix)

Gọi $\mathbf{y}_t \in \mathbb{R}^{m}$ là vector các biến tại thời điểm $t$, với $p$ độ trễ.
Đầu vào cho VARNN tại thời điểm $t$ là:

$$
\mathbf{x}_t = [\mathbf{y}_{t-1}, \mathbf{y}_{t-2}, \ldots, \mathbf{y}_{t-p}]
$$

Nếu dự báo 1 bước trước cho biến mục tiêu $y_t^{(target)}$:

$$
\hat{y}_{t} = f_{\theta}(\mathbf{x}_t)
$$

### 1.5 Dạng VARNN (VAR-NN) sử dụng trong notebook

Mạng FFNN nhận toàn bộ các lag đã được chọn bởi VAR:

$$
\hat{y}_{t} = \beta_0 + \mathbf{\beta}^{\top} \phi(\mathbf{W}\mathbf{x}_t + \mathbf{b})
$$

với:
- $\mathbf{x}_t$: vector lag của tất cả biến
- $\phi(\cdot)$: hàm kích hoạt
- $\mathbf{W}, \mathbf{b}, \mathbf{\beta}$: tham số mạng

### 1.6 Hàm mất mát

Mô hình tối ưu theo MSE:

$$
\mathcal{L}(\theta) = \frac{1}{N} \sum_{t=1}^{N} (y_t - \hat{y}_t)^2
$$


In [1]:

# =========================
# 2. Cấu hình
# =========================
CSV_PATH = './data/processed/cpi_forecast_selected_variables.csv'      # Đổi thành đường dẫn file CSV của bạn
DATE_COL = 'date'
TARGET_COL = 'cpi_mom_inflation'
HORIZON = 1                     # dự báo 1 bước tới
TEST_SIZE = 0.15                # tỷ lệ test cuối chuỗi
VAL_SIZE = 0.15                 # tỷ lệ validation cuối chuỗi (nằm trước test)
ALPHA = 0.05                    # mức ý nghĩa cho ADF
MAX_LAGS = 12                   # monthly data: thường thử 6-12-18; ở đây dùng 12
PPI_CORR_THRESHOLD = 0.10       # nếu |corr| thấp hơn ngưỡng và p-value cao -> cân nhắc loại
PPI_PVALUE_THRESHOLD = 0.10
AUTO_DROP_PPI_IF_WEAK = True
SEED = 42
VERBOSE_FIT = 0                 # 0 để output gọn, 1 nếu muốn xem log train
TARGET_IS_RATE = True           # cpi_mom_inflation đã là MoM (%)


In [2]:

# =========================
# 3. Import thư viện
# =========================
import warnings
warnings.filterwarnings('ignore')

import os
import random
import numpy as np
import pandas as pd

from scipy.stats import pearsonr
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import VAR

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


In [3]:

# =========================
# 4. Đọc dữ liệu và kiểm tra cơ bản
# =========================
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f'Không tìm thấy file: {CSV_PATH}. Hãy đổi biến CSV_PATH cho đúng.')

raw = pd.read_csv(CSV_PATH)
raw[DATE_COL] = pd.to_datetime(raw[DATE_COL])
raw = raw.sort_values(DATE_COL).drop_duplicates(subset=[DATE_COL]).set_index(DATE_COL)

# ép tần suất tháng (month-start); nếu dữ liệu của bạn là cuối tháng, có thể đổi sang 'M'
raw = raw.asfreq('MS')

print('Kích thước dữ liệu gốc:', raw.shape)
print('5 dòng đầu:')
print(raw.head())
print('Thông tin missing values:')
print(raw.isna().sum().sort_values(ascending=False))


Kích thước dữ liệu gốc: (360, 9)
5 dòng đầu:
            cpi_mom_inflation  broad_money  ppi_qoq    wti        gold  policy_rate  VNINDEX     NIKKEI225   USDVND
date                                                                                                               
1995-01-01                3.8        3.567    -0.73  18.39  278.299988         10.8   101.55  18649.820312  11039.0
1995-02-01                3.4        3.303    -0.73  18.49  278.299988         10.8   101.55  17053.429688  11050.0
1995-03-01                0.2        3.495    -0.73  19.17  278.299988         10.8   101.55  16139.950195  11045.0
1995-04-01                1.0        3.434    -0.73  20.38  278.299988         10.8   101.55  16806.750000  11041.0
1995-05-01                1.8        3.247    -0.73  18.89  278.299988         10.8   101.55  15436.790039  11045.0
Thông tin missing values:
cpi_mom_inflation    0
broad_money          0
ppi_qoq              0
wti                  0
gold                 0
po


## 2. Chia train / validation / test theo thời gian

Chia dữ liệu theo thời gian để tránh **data leakage**:
- **train**: dùng để xác định biến đổi, fit scaler, chọn lag, huấn luyện
- **validation**: dùng cho early stopping / điều chỉnh overfit-underfit
- **test**: chỉ dùng đánh giá cuối cùng


In [4]:

# =========================
# 5. Chia train/val/test theo thời gian
# =========================
N = len(raw)
test_n = max(1, int(np.floor(N * TEST_SIZE)))
val_n = max(1, int(np.floor(N * VAL_SIZE)))
train_n = N - val_n - test_n

if train_n <= MAX_LAGS + 10:
    raise ValueError('Tập train quá ngắn so với MAX_LAGS. Hãy giảm MAX_LAGS hoặc tăng dữ liệu.')

train_raw = raw.iloc[:train_n].copy()
val_raw = raw.iloc[train_n: train_n + val_n].copy()
test_raw = raw.iloc[train_n + val_n:].copy()

print('Số quan sát:')
print('Train:', train_raw.shape)
print('Validation:', val_raw.shape)
print('Test:', test_raw.shape)
print('Mốc thời gian:')
print('Train :', train_raw.index.min().date(), '->', train_raw.index.max().date())
print('Val   :', val_raw.index.min().date(), '->', val_raw.index.max().date())
print('Test  :', test_raw.index.min().date(), '->', test_raw.index.max().date())


Số quan sát:
Train: (252, 9)
Validation: (54, 9)
Test: (54, 9)
Mốc thời gian:
Train : 1995-01-01 -> 2015-12-01
Val   : 2016-01-01 -> 2020-06-01
Test  : 2020-07-01 -> 2024-12-01



## 3. Xử lý `ppi_qoq`: đánh dấu là biến quý, kiểm tra tương quan, và loại nếu yếu

`ppi_qoq` là biến theo **quý (QoQ)** nhưng dữ liệu đang ở tần suất **tháng**. Vì vậy notebook xử lý theo hướng:
1. thêm cờ **quý / cuối quý** để mô hình biết đây là biến thấp tần,
2. kiểm tra tương quan với biến mục tiêu,
3. nếu tương quan yếu thì **tự động loại** để tránh nhiễu.


In [5]:

# =========================
# 6. Xử lý riêng ppi_qoq
# =========================
df = raw.copy()

if 'ppi_qoq' in df.columns:
    # Cờ biến quý / cuối quý
    df['ppi_qoq_is_quarterly'] = 1
    df['is_quarter_end'] = df.index.month.isin([3, 6, 9, 12]).astype(int)
    df['quarter_num'] = df.index.quarter.astype(int)

    # Tương quan trên train để tránh leakage
    tmp_train = df.loc[train_raw.index].copy()
    valid_mask = tmp_train[['ppi_qoq', TARGET_COL]].dropna().index

    if len(valid_mask) >= 8:
        corr_val, corr_p = pearsonr(tmp_train.loc[valid_mask, 'ppi_qoq'], tmp_train.loc[valid_mask, TARGET_COL])
    else:
        corr_val, corr_p = np.nan, np.nan

    print('Đánh giá ppi_qoq trên train:')
    print(f'  Pearson corr với {TARGET_COL}: {corr_val:.4f}' if pd.notna(corr_val) else '  Pearson corr: NaN (không đủ dữ liệu)')
    print(f'  p-value: {corr_p:.4f}' if pd.notna(corr_p) else '  p-value: NaN')

    ppi_drop_reason = None
    if AUTO_DROP_PPI_IF_WEAK and pd.notna(corr_val) and pd.notna(corr_p):
        if abs(corr_val) < PPI_CORR_THRESHOLD and corr_p > PPI_PVALUE_THRESHOLD:
            ppi_drop_reason = f'ppi_qoq bị loại vì |corr|={abs(corr_val):.4f} < {PPI_CORR_THRESHOLD} và p-value={corr_p:.4f} > {PPI_PVALUE_THRESHOLD}'
            df = df.drop(columns=['ppi_qoq'])

    if ppi_drop_reason:
        print('  =>', ppi_drop_reason)
    else:
        print('  => Giữ ppi_qoq và các cờ liên quan (nếu tồn tại)')
else:
    print('Không có cột ppi_qoq trong dữ liệu.')


Đánh giá ppi_qoq trên train:
  Pearson corr: NaN (không đủ dữ liệu)
  p-value: NaN
  => Giữ ppi_qoq và các cờ liên quan (nếu tồn tại)



## 4. Kiểm định tính dừng và xác định phép biến đổi

Chiến lược mặc định:
- `cpi_mom_inflation`: đã là **MoM %**, thường giữ nguyên trước
- `ppi_qoq`: giữ nguyên nếu còn được giữ lại
- `policy_rate`: ưu tiên **sai phân bậc 1** nếu không dừng
- các biến mức dương như `wti`, `gold`, `VNINDEX`, `NIKKEI225`, `USDVND`, `broad_money`: ưu tiên **log-diff** nếu không dừng

> **Nguyên tắc chống leakage:** ADF chỉ được chạy trên **train**, từ đó suy ra phép biến đổi rồi áp lên toàn bộ chuỗi.


In [6]:

# =========================
# 7. Kiểm định ADF và quyết định phép biến đổi trên TRAIN
# =========================

def adf_pvalue(series):
    s = pd.Series(series).dropna()
    if len(s) < 12 or s.nunique() < 3:
        return np.nan
    try:
        return adfuller(s, autolag='AIC')[1]
    except Exception:
        return np.nan

positive_level_candidates = [c for c in ['broad_money', 'wti', 'gold', 'VNINDEX', 'NIKKEI225', 'USDVND'] if c in df.columns]
diff_candidates = [c for c in ['policy_rate'] if c in df.columns]
rate_like_candidates = [c for c in [TARGET_COL, 'ppi_qoq'] if c in df.columns]
binary_or_categorical_keep = [c for c in ['ppi_qoq_is_quarterly', 'is_quarter_end', 'quarter_num'] if c in df.columns]

transform_recipe = {}
report_rows = []

train_current = df.loc[train_raw.index].copy()

for col in df.columns:
    p_before = adf_pvalue(train_current[col])

    if col in binary_or_categorical_keep:
        transform_recipe[col] = 'keep'
    elif col in rate_like_candidates:
        # đã là rate/growth => ưu tiên giữ, chỉ diff nếu thật sự cần thiết
        if pd.notna(p_before) and p_before < ALPHA:
            transform_recipe[col] = 'keep'
        else:
            transform_recipe[col] = 'diff'
    elif col in diff_candidates:
        if pd.notna(p_before) and p_before < ALPHA:
            transform_recipe[col] = 'keep'
        else:
            transform_recipe[col] = 'diff'
    elif col in positive_level_candidates:
        # log-diff nếu không dừng; nếu có giá trị không dương thì fallback sang diff
        if pd.notna(p_before) and p_before < ALPHA:
            transform_recipe[col] = 'keep'
        else:
            if (train_current[col].dropna() > 0).all():
                transform_recipe[col] = 'log_diff'
            else:
                transform_recipe[col] = 'diff'
    else:
        # fallback tổng quát
        if pd.notna(p_before) and p_before < ALPHA:
            transform_recipe[col] = 'keep'
        else:
            transform_recipe[col] = 'diff'

    report_rows.append([col, p_before, transform_recipe[col]])

transform_report = pd.DataFrame(report_rows, columns=['column', 'adf_pvalue_before', 'chosen_transform'])
print(transform_report.sort_values('column').to_string(index=False))


              column  adf_pvalue_before chosen_transform
           NIKKEI225           0.469353         log_diff
              USDVND           0.912014         log_diff
             VNINDEX           0.256874         log_diff
         broad_money           0.760807         log_diff
   cpi_mom_inflation           0.018204             keep
                gold           0.708667         log_diff
      is_quarter_end                NaN             keep
         policy_rate           0.033659             keep
             ppi_qoq                NaN             diff
ppi_qoq_is_quarterly                NaN             keep
         quarter_num           0.000000             keep
                 wti           0.328714         log_diff


In [7]:
# =========================
# 8. Áp dụng phép biến đổi cho toàn bộ chuỗi
#    + làm sạch feature
#    + seasonal encoding đúng
# =========================

def apply_transform(series, recipe):
    s = series.copy()
    if recipe == 'keep':
        return s
    elif recipe == 'diff':
        return s.diff()
    elif recipe == 'log_diff':
        return np.log(s).diff()
    else:
        raise ValueError(f'Recipe không hợp lệ: {recipe}')

# --- ép target giữ nguyên nếu bạn đang forecast CPI MoM trực tiếp ---
# Nếu muốn forecast delta(CPI MoM) thì bỏ dòng này.
transform_recipe[TARGET_COL] = 'keep'

transformed = pd.DataFrame(index=df.index)
for col in df.columns:
    transformed[col] = apply_transform(df[col], transform_recipe[col])

# =========================
# Làm sạch các cột "cờ" / seasonal
# =========================

# 1) ppi_qoq_is_quarterly luôn = 1 => vô ích, drop
if 'ppi_qoq_is_quarterly' in transformed.columns:
    transformed = transformed.drop(columns=['ppi_qoq_is_quarterly'])

# 2) quarter_num -> seasonal cyclic encoding
if 'quarter_num' in transformed.columns:
    q = df.index.quarter.astype(float)
    transformed['quarter_sin'] = np.sin(2 * np.pi * q / 4)
    transformed['quarter_cos'] = np.cos(2 * np.pi * q / 4)
    transformed = transformed.drop(columns=['quarter_num'])

# 3) is_quarter_end giữ như dummy
if 'is_quarter_end' in transformed.columns:
    transformed['is_quarter_end'] = transformed['is_quarter_end'].ffill().bfill().astype(float)

# =========================
# Xử lý riêng ppi_qoq
# =========================
# Nếu ppi_qoq tồn tại nhưng sau transform gần constant / ADF NaN / quá ít variation
# thì drop để tránh nhiễu trong bài toán monthly forecast
if 'ppi_qoq' in transformed.columns:
    tmp_ppi_train = transformed.loc[transformed.index <= train_raw.index.max(), 'ppi_qoq'].dropna()
    ppi_adf_after = adf_pvalue(tmp_ppi_train)

    if tmp_ppi_train.nunique() < 3 or pd.isna(ppi_adf_after):
        print("Drop ppi_qoq vì sau transform chuỗi gần constant / ADF không có ý nghĩa cho monthly modeling.")
        transformed = transformed.drop(columns=['ppi_qoq'])

# Drop toàn bộ NaN sinh ra do diff/log_diff
transformed = transformed.dropna().copy()

# Resplit theo mốc thời gian sau transform
train = transformed.loc[transformed.index <= train_raw.index.max()].copy()
val = transformed.loc[
    (transformed.index > train_raw.index.max()) &
    (transformed.index <= val_raw.index.max())
].copy()
test = transformed.loc[transformed.index > val_raw.index.max()].copy()

print('Kích thước sau transform:')
print('Train:', train.shape)
print('Val  :', val.shape)
print('Test :', test.shape)

print('\nADF sau transform (trên train):')
post_rows = []
for col in train.columns:
    # Skip ADF cho dummy/seasonal deterministic
    if col in ['is_quarter_end', 'quarter_sin', 'quarter_cos']:
        post_rows.append([col, 'SKIP_DUMMY_OR_SEASONAL'])
    else:
        post_rows.append([col, adf_pvalue(train[col])])

post_adf = pd.DataFrame(post_rows, columns=['column', 'adf_pvalue_after'])
print(post_adf.sort_values('column').to_string(index=False))

print('\nDanh sách cột sau làm sạch:')
print(list(train.columns))

Drop ppi_qoq vì sau transform chuỗi gần constant / ADF không có ý nghĩa cho monthly modeling.
Kích thước sau transform:
Train: (251, 11)
Val  : (54, 11)
Test : (54, 11)

ADF sau transform (trên train):
           column       adf_pvalue_after
        NIKKEI225                    0.0
           USDVND               0.065607
          VNINDEX               0.000015
      broad_money                    0.0
cpi_mom_inflation               0.009519
             gold                    0.0
   is_quarter_end SKIP_DUMMY_OR_SEASONAL
      policy_rate               0.033627
      quarter_cos SKIP_DUMMY_OR_SEASONAL
      quarter_sin SKIP_DUMMY_OR_SEASONAL
              wti                    0.0

Danh sách cột sau làm sạch:
['cpi_mom_inflation', 'broad_money', 'wti', 'gold', 'policy_rate', 'VNINDEX', 'NIKKEI225', 'USDVND', 'is_quarter_end', 'quarter_sin', 'quarter_cos']



## 5. Chọn lag bằng VAR

Ta dùng VAR trên **tập train đã transform** để chọn số lag $p$ bằng tiêu chí thông tin (AIC/BIC/HQIC). Ở đây mặc định ưu tiên **BIC** nếu có để tránh mô hình quá phức tạp, nếu không thì fallback sang **AIC**.


In [8]:
# =========================
# 9. Chọn lag bằng VAR (chỉ dùng các biến endogenous thật)
# =========================

# Các cột không nên đưa vào VAR để chọn lag:
# - dummy / seasonal deterministic
exclude_from_var = [
    'is_quarter_end',
    'quarter_sin',
    'quarter_cos'
]

# Candidate columns cho VAR
candidate_var_cols = [c for c in train.columns if c not in exclude_from_var]

# Loại cột constant / near-constant
candidate_var_cols = [c for c in candidate_var_cols if train[c].nunique(dropna=True) > 1]

# Chỉ giữ cột stationary theo ADF
stationary_var_cols = []
non_stationary_var_cols = []

for col in candidate_var_cols:
    pval = adf_pvalue(train[col])
    if pd.notna(pval) and pval < ALPHA:
        stationary_var_cols.append(col)
    else:
        non_stationary_var_cols.append(col)

print("Các cột stationary dùng cho VAR chọn lag:")
print(stationary_var_cols)

if non_stationary_var_cols:
    print("\nCác cột bị loại khỏi VAR vì chưa stationary:")
    print(non_stationary_var_cols)

# Fallback nếu lọc quá chặt
if len(stationary_var_cols) < 2:
    print("\nCảnh báo: số cột stationary < 2, fallback sang candidate_var_cols.")
    var_cols = candidate_var_cols
else:
    var_cols = stationary_var_cols

if len(var_cols) < 2:
    raise ValueError("VAR cần ít nhất 2 biến endogenous. Hãy kiểm tra lại transform/ADF.")

train_var = train[var_cols].copy()

# maxlags hiệu dụng
maxlags_effective = max(1, min(MAX_LAGS, max(1, len(train_var) // 5)))

# Fit VAR để chọn lag
var_model = VAR(train_var)
lag_order_res = var_model.select_order(maxlags=maxlags_effective)

selected_orders = lag_order_res.selected_orders
print("\nLag order selection:", selected_orders)

# Ưu tiên BIC, fallback AIC
if selected_orders.get('bic', None) is not None:
    p_selected_raw = int(selected_orders['bic'])
elif selected_orders.get('aic', None) is not None:
    p_selected_raw = int(selected_orders['aic'])
else:
    p_selected_raw = 1

# Heuristic nhẹ cho monthly macro:
# tránh p = 0 hoặc quá ngắn
p_selected = p_selected_raw

print("Lag p raw từ tiêu chí thông tin:", p_selected_raw)
print("Lag p dùng cho VARNN:", p_selected)

# In thêm candidate lag để bạn cân nhắc CV lag nếu muốn
lag_candidates = sorted(set([
    int(v) for v in selected_orders.values()
    if v is not None and not pd.isna(v)
]))
print("Lag candidates từ AIC/BIC/HQIC/FPE:", lag_candidates)


Các cột stationary dùng cho VAR chọn lag:
['cpi_mom_inflation', 'broad_money', 'wti', 'gold', 'policy_rate', 'VNINDEX', 'NIKKEI225']

Các cột bị loại khỏi VAR vì chưa stationary:
['USDVND']

Lag order selection: {'aic': np.int64(1), 'bic': np.int64(1), 'hqic': np.int64(1), 'fpe': np.int64(1)}
Lag p raw từ tiêu chí thông tin: 1
Lag p dùng cho VARNN: 1
Lag candidates từ AIC/BIC/HQIC/FPE: [1]



## 6. Chuẩn hóa dữ liệu và tạo supervised matrix

- **Scaler chỉ fit trên train**.
- Với supervised matrix, đầu vào tại thời điểm $t$ là toàn bộ lag của tất cả biến từ $t-p$ đến $t-1$.
- Mục tiêu mặc định: dự báo `cpi_mom_inflation` tại $t + h - 1$, với $h = 1$.


In [9]:

# =========================
# 10. StandardScaler (fit trên train) + build full scaled dataframe
# =========================
all_transformed = pd.concat([train, val, test], axis=0)

feature_cols = list(all_transformed.columns)
if TARGET_COL not in feature_cols:
    raise KeyError(f'Không tìm thấy cột mục tiêu {TARGET_COL} sau transform.')

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(train[feature_cols])
y_train_scaled = scaler_y.fit_transform(train[[TARGET_COL]])

all_X_scaled = pd.DataFrame(
    scaler_X.transform(all_transformed[feature_cols]),
    index=all_transformed.index,
    columns=feature_cols
)
all_y_scaled = pd.Series(
    scaler_y.transform(all_transformed[[TARGET_COL]]).ravel(),
    index=all_transformed.index,
    name=TARGET_COL
)

print('Số features sau preprocess:', len(feature_cols))
print('Danh sách features:', feature_cols)


Số features sau preprocess: 11
Danh sách features: ['cpi_mom_inflation', 'broad_money', 'wti', 'gold', 'policy_rate', 'VNINDEX', 'NIKKEI225', 'USDVND', 'is_quarter_end', 'quarter_sin', 'quarter_cos']


In [10]:

# =========================
# 11. Tạo supervised matrix
# =========================

def make_supervised(X_df, y_series, p, horizon=1):
    X_list, y_list, dates_list = [], [], []
    values_X = X_df.values
    values_y = y_series.values
    idx = X_df.index

    for i in range(p, len(X_df) - horizon + 1):
        x_i = values_X[i-p:i].reshape(-1)
        y_i = values_y[i + horizon - 1]
        d_i = idx[i + horizon - 1]
        X_list.append(x_i)
        y_list.append(y_i)
        dates_list.append(d_i)

    return np.asarray(X_list), np.asarray(y_list).reshape(-1, 1), pd.DatetimeIndex(dates_list)

X_all, y_all, y_dates = make_supervised(all_X_scaled, all_y_scaled, p_selected, HORIZON)

train_end = train.index.max()
val_end = val.index.max()

mask_train = y_dates <= train_end
mask_val = (y_dates > train_end) & (y_dates <= val_end)
mask_test = y_dates > val_end

X_train, y_train = X_all[mask_train], y_all[mask_train]
X_val, y_val = X_all[mask_val], y_all[mask_val]
X_test, y_test = X_all[mask_test], y_all[mask_test]
dates_train, dates_val, dates_test = y_dates[mask_train], y_dates[mask_val], y_dates[mask_test]

print('Shape supervised matrix:')
print('X_train:', X_train.shape, 'y_train:', y_train.shape)
print('X_val  :', X_val.shape, 'y_val  :', y_val.shape)
print('X_test :', X_test.shape, 'y_test :', y_test.shape)


Shape supervised matrix:
X_train: (250, 11) y_train: (250, 1)
X_val  : (54, 11) y_val  : (54, 1)
X_test : (54, 11) y_test : (54, 1)



## 7. Baseline ngây thơ (naive) để so sánh

Baseline dùng **giá trị mục tiêu ở lag 1** sau transform làm dự báo cho bước tiếp theo:

$$
\hat{y}_{t}^{naive} = y_{t-1}
$$

Nếu VARNN không vượt được baseline này, đó là tín hiệu underfitting hoặc pipeline chưa phù hợp.


In [11]:

# =========================
# 12. Baseline naive forecast trên không gian đã transform (trước inverse-scale)
# =========================
# Vì X_t chứa lag của tất cả biến, lag gần nhất của TARGET_COL nằm ở block cuối cùng trong cửa sổ thời gian? 
# Ở đây dễ nhất là lấy trực tiếp từ chuỗi mục tiêu đã scale theo ngày.

from sklearn.metrics import root_mean_squared_error


lag1_target_scaled = all_y_scaled.shift(1)
naive_preds = lag1_target_scaled.reindex(y_dates).values.reshape(-1, 1)

naive_test = naive_preds[mask_test]
naive_train = naive_preds[mask_train]

# inverse transform về đơn vị gốc của target sau transform
true_train_inv = scaler_y.inverse_transform(y_train)
true_test_inv = scaler_y.inverse_transform(y_test)
naive_train_inv = scaler_y.inverse_transform(naive_train)
naive_test_inv = scaler_y.inverse_transform(naive_test)
baseline_train_rmse = root_mean_squared_error(true_train_inv, naive_train_inv)
baseline_test_rmse = root_mean_squared_error(true_test_inv, naive_test_inv)

print('Baseline naive RMSE:')
print('Train:', round(float(baseline_train_rmse), 6))
print('Test :', round(float(baseline_test_rmse), 6))


Baseline naive RMSE:
Train: 0.831199
Test : 0.516403



## 8. Định nghĩa mô hình VARNN (FFNN với lag chọn từ VAR)

Mạng dưới đây là một thực thi thực dụng của VARNN:
- input dimension = $p \times m$
- 1 hoặc 2 hidden layers
- hỗ trợ dropout, L2 regularization, learning rate tuning
- dùng **EarlyStopping** + **ReduceLROnPlateau**


In [12]:

# =========================
# 13. Định nghĩa mô hình VARNN
# =========================

def build_varnn(input_dim, units1=64, units2=32, dropout=0.1, l2_reg=1e-4, lr=1e-3, activation='relu'):
    inputs = keras.Input(shape=(input_dim,))
    x = layers.Dense(units1, activation=activation,
                     kernel_regularizer=regularizers.l2(l2_reg))(inputs)
    if dropout and dropout > 0:
        x = layers.Dropout(dropout)(x)

    if units2 and units2 > 0:
        x = layers.Dense(units2, activation=activation,
                         kernel_regularizer=regularizers.l2(l2_reg))(x)
        if dropout and dropout > 0:
            x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(1, activation='linear')(x)
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='mse',
        metrics=[keras.metrics.RootMeanSquaredError(name='rmse'), keras.metrics.MeanAbsoluteError(name='mae')]
    )
    return model


def inverse_y(y_scaled):
    y_scaled = np.asarray(y_scaled).reshape(-1, 1)
    return scaler_y.inverse_transform(y_scaled)


def regression_report(y_true_scaled, y_pred_scaled, prefix=''):
    y_true_inv = inverse_y(y_true_scaled)
    y_pred_inv = inverse_y(y_pred_scaled)
    rmse = root_mean_squared_error(y_true_inv, y_pred_inv)
    mae = mean_absolute_error(y_true_inv, y_pred_inv)
    mape = np.mean(np.abs((y_true_inv - y_pred_inv) / np.where(np.abs(y_true_inv) < 1e-8, 1e-8, y_true_inv))) * 100
    r2 = r2_score(y_true_inv, y_pred_inv)
    return {
        f'{prefix}RMSE': float(rmse),
        f'{prefix}MAE': float(mae),
        f'{prefix}MAPE': float(mape),
        f'{prefix}R2': float(r2)
    }



## 9. Tối ưu mô hình bằng TimeSeriesSplit Cross-Validation

Ta dùng **TimeSeriesSplit** trên tập **train + validation** để tối ưu siêu tham số mà vẫn giữ trật tự thời gian.

Không dùng random K-Fold cho chuỗi thời gian.


In [13]:

# =========================
# 14. Cross-validation + hyperparameter tuning
# =========================
X_trainval = np.vstack([X_train, X_val]) if len(X_val) > 0 else X_train.copy()
y_trainval = np.vstack([y_train, y_val]) if len(y_val) > 0 else y_train.copy()

print('Train+Val shape:', X_trainval.shape, y_trainval.shape)

param_grid = [
    {'units1': 32,  'units2': 0,  'dropout': 0.00, 'l2_reg': 1e-5, 'lr': 1e-3, 'batch_size': 16, 'epochs': 200, 'activation': 'relu'},
    {'units1': 64,  'units2': 32, 'dropout': 0.10, 'l2_reg': 1e-4, 'lr': 1e-3, 'batch_size': 16, 'epochs': 250, 'activation': 'relu'},
    {'units1': 128, 'units2': 64, 'dropout': 0.15, 'l2_reg': 1e-4, 'lr': 5e-4, 'batch_size': 16, 'epochs': 300, 'activation': 'relu'},
    {'units1': 64,  'units2': 32, 'dropout': 0.20, 'l2_reg': 1e-3, 'lr': 5e-4, 'batch_size': 32, 'epochs': 250, 'activation': 'tanh'},
]

n_splits = min(4, max(2, len(X_trainval) // 24))
tscv = TimeSeriesSplit(n_splits=n_splits)

cv_results = []
input_dim = X_trainval.shape[1]

for p_idx, params in enumerate(param_grid, start=1):
    fold_metrics = []
    print(f'Candidate {p_idx}/{len(param_grid)}: {params} ====')

    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_trainval), start=1):
        X_tr, X_va = X_trainval[tr_idx], X_trainval[va_idx]
        y_tr, y_va = y_trainval[tr_idx], y_trainval[va_idx]

        model = build_varnn(
            input_dim=input_dim,
            units1=params['units1'],
            units2=params['units2'],
            dropout=params['dropout'],
            l2_reg=params['l2_reg'],
            lr=params['lr'],
            activation=params['activation']
        )

        callbacks = [
            keras.callbacks.EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-5)
        ]

        history = model.fit(
            X_tr, y_tr,
            validation_data=(X_va, y_va),
            epochs=params['epochs'],
            batch_size=params['batch_size'],
            verbose=VERBOSE_FIT,
            callbacks=callbacks
        )

        pred_va = model.predict(X_va, verbose=0)
        fold_report = regression_report(y_va, pred_va)
        fold_metrics.append(fold_report)

        print(f'Fold {fold}: RMSE={fold_report["RMSE"]:.6f}, MAE={fold_report["MAE"]:.6f}, MAPE={fold_report["MAPE"]:.4f}%, R2={fold_report["R2"]:.6f}')

    avg_rmse = float(np.mean([m['RMSE'] for m in fold_metrics]))
    avg_mae = float(np.mean([m['MAE'] for m in fold_metrics]))
    avg_mape = float(np.mean([m['MAPE'] for m in fold_metrics]))
    avg_r2 = float(np.mean([m['R2'] for m in fold_metrics]))

    result_row = {
        'params': params,
        'cv_RMSE': avg_rmse,
        'cv_MAE': avg_mae,
        'cv_MAPE': avg_mape,
        'cv_R2': avg_r2,
    }
    cv_results.append(result_row)

cv_df = pd.DataFrame(cv_results).sort_values(['cv_RMSE', 'cv_MAE']).reset_index(drop=True)
print(' Kết quả CV tổng hợp =====')
print(cv_df.to_string(index=False))

best_params = cv_df.iloc[0]['params']
print('Best params:', best_params)


Train+Val shape: (304, 11) (304, 1)
Candidate 1/4: {'units1': 32, 'units2': 0, 'dropout': 0.0, 'l2_reg': 1e-05, 'lr': 0.001, 'batch_size': 16, 'epochs': 200, 'activation': 'relu'} ====
Fold 1: RMSE=0.771268, MAE=0.550280, MAPE=280523510.8058%, R2=-0.140957
Fold 2: RMSE=0.995615, MAE=0.678761, MAPE=115.7427%, R2=-0.102572
Fold 3: RMSE=0.670926, MAE=0.485036, MAPE=143.9849%, R2=0.239753
Fold 4: RMSE=0.475498, MAE=0.372097, MAPE=141065528.4118%, R2=-0.074845
Candidate 2/4: {'units1': 64, 'units2': 32, 'dropout': 0.1, 'l2_reg': 0.0001, 'lr': 0.001, 'batch_size': 16, 'epochs': 250, 'activation': 'relu'} ====
Fold 1: RMSE=0.683723, MAE=0.454096, MAPE=264640835.3252%, R2=0.103356
Fold 2: RMSE=0.876184, MAE=0.615874, MAPE=98.9091%, R2=0.146083
Fold 3: RMSE=0.634166, MAE=0.439906, MAPE=144.9684%, R2=0.320776
Fold 4: RMSE=0.471716, MAE=0.366959, MAPE=98257257.7629%, R2=-0.057818
Candidate 3/4: {'units1': 128, 'units2': 64, 'dropout': 0.15, 'l2_reg': 0.0001, 'lr': 0.0005, 'batch_size': 16, 'epoch


## 10. Huấn luyện mô hình cuối cùng

Mô hình cuối cùng được fit trên **train + validation** và đánh giá trên **test**.
Dùng phần cuối của train+validation làm validation nội bộ cho EarlyStopping.


In [14]:

# =========================
# 15. Huấn luyện mô hình cuối cùng
# =========================
# Tạo validation nội bộ ở cuối chuỗi train+val
inner_val_size = max(1, int(len(X_trainval) * 0.10))
X_tr_final, X_va_final = X_trainval[:-inner_val_size], X_trainval[-inner_val_size:]
y_tr_final, y_va_final = y_trainval[:-inner_val_size], y_trainval[-inner_val_size:]

final_model = build_varnn(
    input_dim=input_dim,
    units1=best_params['units1'],
    units2=best_params['units2'],
    dropout=best_params['dropout'],
    l2_reg=best_params['l2_reg'],
    lr=best_params['lr'],
    activation=best_params['activation']
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=12, min_lr=1e-5)
]

history = final_model.fit(
    X_tr_final, y_tr_final,
    validation_data=(X_va_final, y_va_final),
    epochs=best_params['epochs'],
    batch_size=best_params['batch_size'],
    verbose=VERBOSE_FIT,
    callbacks=callbacks
)

pred_trainval = final_model.predict(X_trainval, verbose=0)
pred_test = final_model.predict(X_test, verbose=0)

trainval_report = regression_report(y_trainval, pred_trainval, prefix='trainval_')
test_report = regression_report(y_test, pred_test, prefix='test_')

print('===== Final model report =====')
for k, v in {**trainval_report, **test_report}.items():
    print(f'{k}: {v:.6f}')


===== Final model report =====
trainval_RMSE: 0.665217
trainval_MAE: 0.459665
trainval_MAPE: 115621728.799470
trainval_R2: 0.308090
test_RMSE: 0.456869
test_MAE: 0.348790
test_MAPE: 149851849.339413
test_R2: -0.468626



## 11. Chẩn đoán underfitting / overfitting và khắc phục tự động

### Tiêu chí thực dụng trong notebook
- **Underfitting** nếu mô hình không vượt baseline đáng kể trên cả train+val và test.
- **Overfitting** nếu lỗi test lớn hơn nhiều so với train+val.

### Cách khắc phục
- **Underfit**:
  - tăng số units,
  - thêm hidden layer hoặc tăng lag,
  - giảm regularization/dropout,
  - train lâu hơn.
- **Overfit**:
  - giảm độ phức tạp mô hình,
  - tăng dropout/L2,
  - dùng early stopping mạnh hơn,
  - giảm số feature hoặc lag.


In [15]:
# =========================
# 16. Chẩn đoán và tự động khắc phục
#     KHÔNG dùng test để chọn mô hình
# =========================

def diagnose_fit(train_rmse, val_rmse, baseline_train_rmse):
    """
    - underfit: train chưa vượt baseline đủ rõ
    - overfit: val lỗi lớn hơn train quá nhiều
    """
    if train_rmse >= 0.98 * baseline_train_rmse:
        return 'underfit'
    elif val_rmse > 1.25 * train_rmse:
        return 'overfit'
    else:
        return 'balanced'


# Đánh giá model hiện tại trên validation nội bộ (KHÔNG phải test)
base_val_pred = final_model.predict(X_va_final, verbose=0)
base_val_report = regression_report(y_va_final, base_val_pred, prefix='val_')

print("===== Base model report (inner validation) =====")
for k, v in base_val_report.items():
    print(f'{k}: {v:.6f}')

issue = diagnose_fit(
    train_rmse=trainval_report['trainval_RMSE'],
    val_rmse=base_val_report['val_RMSE'],
    baseline_train_rmse=baseline_train_rmse
)

print('\nChẩn đoán hiện tại:', issue)

remedy_params = None
if issue == 'underfit':
    remedy_params = best_params.copy()
    remedy_params['units1'] = max(remedy_params['units1'], 128)
    remedy_params['units2'] = max(remedy_params['units2'], 64)
    remedy_params['dropout'] = max(0.0, remedy_params['dropout'] - 0.05)
    remedy_params['l2_reg'] = max(1e-6, remedy_params['l2_reg'] / 10)
    remedy_params['epochs'] = int(remedy_params['epochs'] * 1.25)
    print('Áp dụng remedy cho underfit:', remedy_params)

elif issue == 'overfit':
    remedy_params = best_params.copy()
    remedy_params['units1'] = max(16, remedy_params['units1'] // 2)
    remedy_params['units2'] = 0 if remedy_params['units2'] == 0 else max(8, remedy_params['units2'] // 2)
    remedy_params['dropout'] = min(0.35, remedy_params['dropout'] + 0.10)
    remedy_params['l2_reg'] = min(1e-2, remedy_params['l2_reg'] * 10)
    remedy_params['epochs'] = int(remedy_params['epochs'] * 0.90)
    print('Áp dụng remedy cho overfit:', remedy_params)

else:
    print('Mô hình cân bằng tương đối; không cần remedy tự động.')

best_model = final_model
selected_config = dict(best_params)

# So sánh remedy trên validation nội bộ, KHÔNG dùng test
if remedy_params is not None:
    remedy_model = build_varnn(
        input_dim=input_dim,
        units1=remedy_params['units1'],
        units2=remedy_params['units2'],
        dropout=remedy_params['dropout'],
        l2_reg=remedy_params['l2_reg'],
        lr=remedy_params['lr'],
        activation=remedy_params['activation']
    )

    remedy_callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-5)
    ]

    remedy_model.fit(
        X_tr_final, y_tr_final,
        validation_data=(X_va_final, y_va_final),
        epochs=remedy_params['epochs'],
        batch_size=remedy_params['batch_size'],
        verbose=VERBOSE_FIT,
        callbacks=remedy_callbacks
    )

    remedy_val_pred = remedy_model.predict(X_va_final, verbose=0)
    remedy_val_report = regression_report(y_va_final, remedy_val_pred, prefix='val_')

    print('\n===== Remedy model report (inner validation) =====')
    for k, v in remedy_val_report.items():
        print(f'{k}: {v:.6f}')

    if remedy_val_report['val_RMSE'] < base_val_report['val_RMSE']:
        best_model = remedy_model
        selected_config = remedy_params
        print('\n=> Remedy model tốt hơn trên validation, chọn remedy model làm mô hình cuối.')
    else:
        print('\n=> Remedy model không tốt hơn trên validation, giữ mô hình trước đó.')

===== Base model report (inner validation) =====
val_RMSE: 0.532270
val_MAE: 0.395673
val_MAPE: 175.877097
val_R2: 0.135428

Chẩn đoán hiện tại: balanced
Mô hình cân bằng tương đối; không cần remedy tự động.



## 12. In kết quả cuối cùng ra màn hình

Bao gồm:
- cấu hình mô hình cuối
- metric train+val và test
- so sánh với baseline
- bảng dự báo vs thực tế cho vài quan sát cuối


In [16]:

# =========================
# 17. Kết quả cuối cùng
# =========================
final_pred_test = best_model.predict(X_test, verbose=0)
final_test_inv = inverse_y(y_test).ravel()
final_pred_test_inv = inverse_y(final_pred_test).ravel()

summary = {
    'p_selected': p_selected,
    'input_dim': input_dim,
    'selected_config': selected_config,
    'baseline_train_RMSE': float(baseline_train_rmse),
    'baseline_test_RMSE': float(baseline_test_rmse),
    'final_test_RMSE': float(root_mean_squared_error(final_test_inv, final_pred_test_inv)),
    'final_test_MAE': float(mean_absolute_error(final_test_inv, final_pred_test_inv)),
    'final_test_R2': float(r2_score(final_test_inv, final_pred_test_inv)),
}

print('===== TÓM TẮT CUỐI CÙNG =====')
for k, v in summary.items():
    print(f'{k}: {v}')

results_df = pd.DataFrame({
    'date': dates_test,
    'actual': final_test_inv,
    'predicted': final_pred_test_inv,
    'abs_error': np.abs(final_test_inv - final_pred_test_inv)
}).set_index('date')

print('===== 10 dự báo cuối cùng =====')
print(results_df.tail(10).to_string())


===== TÓM TẮT CUỐI CÙNG =====
p_selected: 1
input_dim: 11
selected_config: {'units1': 64, 'units2': 32, 'dropout': 0.2, 'l2_reg': 0.001, 'lr': 0.0005, 'batch_size': 32, 'epochs': 250, 'activation': 'tanh'}
baseline_train_RMSE: 0.8311991337820318
baseline_test_RMSE: 0.516403158609843
final_test_RMSE: 0.456869338511755
final_test_MAE: 0.348789848077491
final_test_R2: -0.4686257455837646
===== 10 dự báo cuối cùng =====
            actual  predicted  abs_error
date                                    
2024-03-01   -0.23   0.694127   0.924127
2024-04-01    0.07   0.140593   0.070593
2024-05-01    0.05   0.045751   0.004249
2024-06-01    0.17  -0.049602   0.219602
2024-07-01    0.48   0.130417   0.349583
2024-08-01    0.00   0.502295   0.502295
2024-09-01    0.29   0.292127   0.002127
2024-10-01    0.33   0.393716   0.063716
2024-11-01    0.13   0.466465   0.336465
2024-12-01    0.29   0.436854   0.146854



## 13. Gợi ý tinh chỉnh thêm (nếu cần)

### Khi underfit kéo dài
- Tăng `MAX_LAGS` để VAR có nhiều lựa chọn hơn
- Mở rộng `param_grid`
- Cho phép `units2` lớn hơn
- Giảm `dropout`, giảm `l2_reg`
- Thêm feature có ý nghĩa kinh tế (ví dụ biến ngoại sinh khác)

### Khi overfit kéo dài
- Giảm số feature yếu tương quan
- Tăng ngưỡng loại `ppi_qoq`
- Giảm `p_selected` bằng cách ưu tiên BIC mạnh hơn
- Tăng `dropout`, `l2_reg`
- Giảm số epoch hoặc tăng patience của early stopping theo hướng thận trọng

### Nếu muốn đa bước (multi-step)
- Có thể sửa `HORIZON > 1`
- Hoặc chuyển sang recursive forecast / direct multi-horizon outputs
